# Assignment_2 - Gmail Inbox Sentiment

Dependencies:

(Run the following in UV)

uv add install google-api-python-client

uv add google-auth-oauthlib

uv sync --active


Imports

In [4]:
# Chat Model Imports
from langchain_core.messages import HumanMessage, AIMessage
import gradio as gr
from dotenv import load_dotenv
import os
from openai import OpenAI
import json

from typing import Optional
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model

# Google API Imports
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow


Loading Secrets

In [3]:
# Load the environment

%load_ext dotenv
%dotenv ../05_src/.secrets

System Prompt

In [5]:
# System prompt - Designed to help guide the model to be more sensitive to language that a customer service agent would look for.
# Possible Additions: Ask the model to look at the data as if it was a survey


system_prompt = "You are a friendly, helpful, courteous, and professional customer service agent. Your primary task is to carefully understand and interpret customer feedback, including both positive comments—such as what they appreciate about the company, products, or services—and negative feedback, such as dislikes, pain points, or challenges they face. Always maintain a polite and empathetic tone, ensuring the customer feels heard, valued, and respected. You strictly uphold customer privacy and confidentiality, and you never share or disclose any personal information."

# Old Prompt
# system_prompt = "You are a customer service agent who is friendly, helpful, courteous, and professional. Your task is to understand the customers' feedback this includes  everthing with what they like about the company and products and services offered to their disklikes, pain points and challenges. Always maintain a polite tone and ensure that the customer feels valued and understood. You value the customer's privacy and confidentiality, and you never share personal information."

# Load Open AI and Pydantic Class

In [ ]:
client = OpenAI()

llm = init_chat_model("gpt-4o-mini", model_provider="openai")

# GMAIL Output Class
class GMAIL(BaseModel):
    Summary: str=Field(description="a concise and succinct summary of the overall sentiment, no longer than 1000 tokens.")
    Positive_Summary: str=Field(description="a statement, no longer than one paragraph, that summarizes the positive feedback from all the customers.")
    Negative_Summary: str=Field(description="a statement, no longer than one paragraph, that summarizes the negative feedback from all the customers.")
    Tone: str=Field(description="the tone used to produce the summary.")
    Input_Tokens: int=Field(description="The number of input tokens that were used in the response.")
    Output_Tokens: int=Field(description="The number of output tokens that were used in the response.")

structured_llm = llm.with_structured_output(GMAIL)

feedback_details = structured_llm.invoke("Tell me about how customers feel about the company.")

# Load Data

In [ ]:
# Credentials dictionary for GMAIL Data Pull
# Normally would be in a secrets .json file
credentials = {
    "installed": {
        "client_id": "YOUR_CLIENT_ID.apps.googleusercontent.com",
        "project_id": "your-project-id",
        "auth_uri": "https://accounts.google.com/o/oauth2/auth",
        "token_uri": "https://oauth2.googleapis.com/token",
        "auth_provider_x509_cert_url": "https://www.googleapis.com/oauth2/v1/certs",
        "client_secret": "YOUR_CLIENT_SECRET",
        "redirect_uris": [
            "http://localhost"
        ]
    }
}


In [ ]:
# Get Email Data
SCOPES = ['https://www.googleapis.com/auth/gmail.readonly']

# Would use this in a normal secrets file implementation
# This was done to minimize the files needed for the assignment
#flow = InstalledAppFlow.from_client_secrets_file('credentials.json', SCOPES)


flow = InstalledAppFlow.from_client_config(credentials, SCOPES)
creds = flow.run_local_server(port=0)
service = build('gmail', 'v1', credentials=creds)

results = service.users().messages().list(userId='me', maxResults=10).execute()
messages = results.get('messages', [])
for msg in messages:
    msg_detail = service.users().messages().get(userId='me', id=msg['id']).execute()
    print(msg_detail['snippet'])

Please visit this URL to authorize this application: https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=YOUR_CLIENT_ID.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A64233%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fgmail.readonly&state=MXdFWMA31DY2aHgmvl7f6LCDeUHl0c&access_type=offline


Tools

In [ ]:
tools = [
    {
        "type": "function",
        "name": "get_horoscope",
        "description": "Get horoscope for a given zodiac sign.",
        "parameters": {
            "type": "object",
            "properties": {
                "zodiac_sign": {
                    "type": "string",
                    "description": "Zodiac sign e.g. Aries, Taurus",
                }
            },
            "required": ["zodiac_sign"],
            "additionalProperties": False,
        },
        "strict": True,
    },
]

def get_horoscope(zodiac_sign: str) -> str:
    # Dummy implementation for illustration
    horoscope = f"{zodiac_sign}: Today is a great day for new beginnings."
    return horoscope



In [ ]:
input_list = [
    {"role": "user", "content": simple_chat}
]

In [ ]:
response = client.responses.create(
    model="gpt-5",
    tools=tools,
    input=input_list,
)

# Chat Bot

In [ ]:
# simple_chat > app.py > 251029 > 1:27:50
# 251030 - 3:15
# Refer to tool lab

def simple_chat(message: str, history: list[dict]) -> str:
    langchain_messages = []
    for msg in history:
        if msg['role'] == 'user':
            langchain_messages.append(HumanMessage(content=msg['content']))
        elif msg['role'] == 'assistant':
            langchain_messages.append(AIMessage(content=msg['content']))
    langchain_messages.append(HumanMessage(content=message))

    response = llm.invoke(langchain_messages)

    return response.content

    
gr.ChatInterface(
    fn=simple_chat,
    type="messages"
).launch()

# Semantic Query

# Function Calling or Web Search